In [0]:
import pandas as pd
import numpy as np

#Leyendo la tabla de caracteristicas generales, desempleados, ft y ocupados
cgse = spark.read.format("delta").table("workspace.electiva.cgse")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
ft = spark.read.format("delta").table("workspace.electiva.ft")
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados")

# Convertir el DataFrame de Spark a Pandas
cgse = cgse.toPandas()
desempleados=desempleados.toPandas()
ft=ft.toPandas()
ocupados=ocupados.toPandas()

In [0]:
# Solución del punto uno del taller

In [0]:
# Partimos con la tabla de Caracteristicas generales, seguridad social en salud y educacion (cgse) que contiene la información general de cada colombiano, sin embargo, para el proposito de resolver los puntos solicitados, necesitamos información adicional del mercado laboral. Usamos como llaves del cgse los parametros 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN', que nos permite encajar las columnas de las otras tablas correctamente en la tabla cgse. Para ello creamos una variable llamada llaves con la lista de 'DIRECTORIO', 'SECUENCIA_P', 'ORDEN' para no tener que escribir esta lista cada vez que se haga uso del comando pd.merge().

llaves = ['DIRECTORIO', 'SECUENCIA_P', 'ORDEN']

# Con el comando pd.merge() introducimos en la tabla cgse las columnas PET y FT de la tabla ft. Al introducir esta columna tenemos acceso a los datos de la población en edad de trabajar y la económicamente activa

cgse = pd.merge(cgse, ft[llaves + ['PET', 'FT']], on=llaves, how='left')

# Ahora introducimos en la tabla cgse las columnas DSI de desocupados de la tabla desempleados

cgse = pd.merge(cgse, desempleados[llaves + ['DSI']], on=llaves, how='left')

# Ahora introducimos en la tabla cgse las columnas de la población ocupada y la de ingresos laborales (OCI,INGLABO) de la tabla ocupados

cgse = pd.merge(cgse, ocupados[llaves + ['OCI', 'INGLABO']], on=llaves, how='left')

# Como habran valores vacíos en las columnas DSI y OCI de la tabla de ocupados y desempleados, reemplazamos estos valores por 0, estos nos permitirá realizar operaciones matemáticas con estas columnas

cgse['DSI'] = cgse['DSI'].fillna(0)
cgse['OCI'] = cgse['OCI'].fillna(0)



In [0]:
# Calculo de la población en edad de trabajar a partir del total nacional. Se toma de la tabla cgse la variable P6040 que contiene la información de la edad de la población; se procede a encontrar aquellos que tengan quince o más años y se le aplica el factor de expanción. El resultado de la población en edad de trabajar a junio de 2026 es 41.202.631 personas en Colombia

PET = cgse[cgse['P6040'] >= 15]['FEX_C18'].sum()
print(f"Población en Edad de Trabajar (PET): {PET:,.0f} personas.")

In [0]:
# El calculo de la población que NO está en edad de trabajar se realiza de la misma manera que el cálculo de la población en edad de trabajar, solo que comparamos la variable P6040 con aquellas personas que tienen menos de quince años.

NO_PET = cgse[cgse['P6040'] < 15]['FEX_C18'].sum()
print(f"Población que NO está en Edad de Trabajar: {NO_PET:,.0f} personas.")

In [0]:
# Cálculo de la Población Económicamente Activa (PEA) a nivel nacional. Se toman de la tabla cgse las personas vinculadas a la fuerza de trabajo (FT == 1) y se realiza la sumatoria de su factor de expansión (FEX_C18).

PEA = cgse[cgse['FT'] == 1]['FEX_C18'].sum()
print(f"Población Económicamente Activa (PEA): {PEA:,.0f} personas.")

In [0]:
# Cálculo de la Población Económicamente Inactiva (PEI) a nivel nacional. Corresponde a la Población Fuera de la Fuerza de Trabajo (PFFT), obtenida al restar la fuerza de trabajo activa (PEA) del total de la población en edad de trabajar (PET).

PEI = PET - PEA
print(f"Población Económicamente Inactiva (PEI): {PEI:,.0f} personas.")

In [0]:
# Para hallar la tasa de ocupación y desocupación calculamos primero la cantidad total de personas ocupadas a nivel nacional que se encuentra en la columna OCI, el comparador lógico con 1 nos garantiza que sea afirmativa. Aplicamos luego la fórmula del DANE (Ocupados / PET) * 100 y (Desocupados / PEA) * 100. Hay que tener presente que Desocupados se puede hallar como (PEA - Ocupados)

Ocupados = cgse[cgse['OCI'] == 1]['FEX_C18'].sum()
tasa_ocupacion = (Ocupados / PET) * 100
tasa_desempleo = ((PEA - Ocupados) / PEA) * 100

print(f"• Tasa de Ocupación (TO): {tasa_ocupacion:.2f}%")
print(f"• Tasa de Desempleo (TD): {tasa_desempleo:.2f}%")

In [0]:
# Solución del punto dos del taller

In [0]:
# 1. Crear internamente las variables expandidas por el peso poblacional
cgse['Desocupados_Exp'] = np.where(cgse['DSI'] == 1, 1, 0) * cgse['FEX_C18']
cgse['PEA_Exp'] = np.where(cgse['FT'] == 1, 1, 0) * cgse['FEX_C18']

# 2. Diccionario oficial del DANE (DIVIPOLA) para mapear nombres de departamentos
dic_dptos = {
    '05': 'Antioquia', '08': 'Atlántico', '11': 'Bogotá D.C.', '13': 'Bolívar', '15': 'Boyacá',
    '17': 'Caldas', '18': 'Caquetá', '19': 'Cauca', '20': 'Cesar', '23': 'Córdoba',
    '25': 'Cundinamarca', '27': 'Chocó', '41': 'Huila', '44': 'La Guajira', '47': 'Magdalena',
    '50': 'Meta', '52': 'Nariño', '54': 'Norte de Santander', '63': 'Quindío', '66': 'Risaralda',
    '68': 'Santander', '70': 'Sucre', '73': 'Tolima', '76': 'Valle del Cauca', '81': 'Arauca',
    '85': 'Casanare', '86': 'Putumayo', '88': 'San Andrés', '91': 'Amazonas', '94': 'Guainía',
    '95': 'Guaviare', '97': 'Vaupés', '99': 'Vichada'
}

# =====================================================================
# PARTE A: IDENTIFICACIÓN DE EXTREMOS (MAYOR Y MENOR DESEMPLEO GENERAL)
# =====================================================================
# Agrupamos solo por departamento para conocer la situación macro de cada región
dpto_general = cgse.groupby('DPTO')[['Desocupados_Exp', 'PEA_Exp']].sum().reset_index()
dpto_general['TD_General'] = (dpto_general['Desocupados_Exp'] / dpto_general['PEA_Exp']) * 100

# Homologar el código a 2 caracteres de texto para cruzar con el diccionario
dpto_general['DPTO'] = dpto_general['DPTO'].astype(float).astype(int).astype(str).str.zfill(2)
dpto_general['Nombre'] = dpto_general['DPTO'].map(dic_dptos)

# Extraer filas con los valores máximos y mínimos
extremo_mayor = dpto_general.loc[dpto_general['TD_General'].idxmax()]
extremo_menor = dpto_general.loc[dpto_general['TD_General'].idxmin()]

print("================================================================")
print("             ANÁLISIS DE EXTREMOS REGIONALES                    ")
print("================================================================")
print(f"• MAYOR DESEMPLEO: {extremo_mayor['Nombre']} (Código {extremo_mayor['DPTO']}) con {extremo_mayor['TD_General']:.2f}%")
print(f"• MENOR DESEMPLEO: {extremo_menor['Nombre']} (Código {extremo_menor['DPTO']}) con {extremo_menor['TD_General']:.2f}%")
print("================================================================\n")


# =====================================================================
# PARTE B: CONSTRUCCIÓN DE LA TABLA REQUERIDA (DEPARTAMENTO Y SEXO)
# =====================================================================
# Agrupamos por Departamento y Variable de Sexo al mismo tiempo
dpto_sexo = cgse.groupby(['DPTO', 'P3271'])[['Desocupados_Exp', 'PEA_Exp']].sum().reset_index()

# Calculamos la tasa de desempleo relativa por grupo
dpto_sexo['Tasa_Desempleo'] = (dpto_sexo['Desocupados_Exp'] / dpto_sexo['PEA_Exp']) * 100

# Formateamos el código de departamento y el Sexo (1=Hombre, 2=Mujer)
dpto_sexo['DPTO'] = dpto_sexo['DPTO'].astype(float).astype(int).astype(str).str.zfill(2)
dpto_sexo['Nombre_Departamento'] = dpto_sexo['DPTO'].map(dic_dptos)
dpto_sexo['Sexo_Texto'] = dpto_sexo['P3271'].map({1: 'Hombre', 2: 'Mujer'})

# Creamos y ordenamos el DataFrame final con la estructura exacta solicitada
tabla_estructurada_p2 = dpto_sexo[['DPTO', 'Nombre_Departamento', 'Sexo_Texto', 'Tasa_Desempleo']].copy()

# Renombramos las columnas para la presentación oficial del parcial
tabla_estructurada_p2.columns = ['Código DANE', 'Nombre del Departamento', 'Sexo', 'Tasa de Desempleo (%)']

# Redondear el indicador a dos decimales para limpieza visual
tabla_estructurada_p2['Tasa de Desempleo (%)'] = tabla_estructurada_p2['Tasa de Desempleo (%)'].round(2)

print("================================================================")
print("     TABLA FINAL: INDICADORES POR DEPARTAMENTO Y GÉNERO         ")
print("================================================================")
print(tabla_estructurada_p2.to_string(index=False))
print("================================================================")